# Week 1 — First-Order Ordinary Differential Equations

> **Differential Equations for Scientists & Engineers**  
> *From separable equations to exact forms — every method derived before it is coded.*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Classify a first-order ODE by type (separable, linear, Bernoulli, exact)
2. Derive and apply the **integrating factor** for linear ODEs
3. Solve **Bernoulli equations** via the substitution $v = y^{1-n}$
4. Test for exactness and construct a **potential function** $F(x,y)$
5. Build a **direction field** renderer from scratch
6. Visualise solution families geometrically


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. Separable ODEs

A first-order ODE is **separable** if it can be written as:

$$\frac{dy}{dx} = g(x)\,h(y)$$

Separating variables and integrating both sides:

$$\int \frac{dy}{h(y)} = \int g(x)\,dx + C$$

### Example 1.1 — Exponential Decay

$$\frac{dy}{dx} = -k\,y, \quad y(0) = y_0$$

Separating: $\dfrac{dy}{y} = -k\,dx$, integrating: $\ln|y| = -kx + C_1$, exponentiating: $y(x) = y_0\,e^{-kx}$.

In [ ]:
x = np.linspace(0, 5, 300)
k_values = [0.3, 0.7, 1.2, 2.0]
y0 = 1.0

fig, ax = plt.subplots(figsize=(8, 4))
colors = cm.plasma(np.linspace(0.2, 0.85, len(k_values)))
for k, c in zip(k_values, colors):
    ax.plot(x, y0 * np.exp(-k * x), color=c, lw=2, label=f'k = {k}')

ax.set_xlabel('x'); ax.set_ylabel('y(x)')
ax.set_title('Exponential Decay — Separable ODE family')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 2. Linear First-Order ODEs and the Integrating Factor

The standard form is:

$$\frac{dy}{dx} + P(x)\,y = Q(x)$$

Multiply both sides by the **integrating factor** $\mu(x) = e^{\int P(x)\,dx}$:

$$\frac{d}{dx}\bigl[\mu(x)\,y\bigr] = \mu(x)\,Q(x)$$

Integrating:

$$\boxed{y(x) = \frac{1}{\mu(x)}\left[\int \mu(x)\,Q(x)\,dx + C\right]}$$

### Implementation — Numerical Integrating Factor Solver

In [ ]:
def integrating_factor_solver(P_func, Q_func, x_span, y0, n=500):
    """
    Solve y' + P(x)y = Q(x) using the integrating factor method.
    Integrals are approximated with the cumulative trapezoidal rule.
    """
    x = np.linspace(x_span[0], x_span[1], n)
    dx = x[1] - x[0]

    P_vals = np.array([P_func(xi) for xi in x])
    log_mu = np.cumsum(P_vals) * dx          # integral of P(x)
    mu = np.exp(log_mu)

    muQ = mu * np.array([Q_func(xi) for xi in x])
    integral_muQ = np.cumsum(muQ) * dx       # integral of mu*Q

    C = mu[0] * y0 - integral_muQ[0]        # apply initial condition
    y = (integral_muQ + C) / mu
    return x, y


# Example: y' + (2/x)y = x^2,  y(1) = 1
# Analytical: y(x) = x^3/5 + 4/(5x^2)
P = lambda x: 2.0 / x
Q = lambda x: x**2
y_exact = lambda x: x**3 / 5.0 + 4.0 / (5.0 * x**2)

x_num, y_num = integrating_factor_solver(P, Q, x_span=(1, 4), y0=1.0)
x_ex = np.linspace(1, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_ex, y_exact(x_ex), 'k-', lw=2, label='Analytical')
axes[0].plot(x_num, y_num, 'r--', lw=1.5, label='Numerical (integrating factor)')
axes[0].set_title("Linear ODE: $y' + (2/x)y = x^2$")
axes[0].legend(frameon=False)

error = np.abs(y_num - y_exact(x_num))
axes[1].semilogy(x_num, error, color='steelblue', lw=2)
axes[1].set_title('Absolute Error')
axes[1].set_ylabel('|y_num - y_exact|')

plt.tight_layout(); plt.show()
print(f"Max absolute error: {error.max():.2e}")

---

## 3. Bernoulli Equations

A **Bernoulli equation** has the form:

$$\frac{dy}{dx} + P(x)\,y = Q(x)\,y^n, \quad n \neq 0, 1$$

The substitution $v = y^{1-n}$ (divide both sides by $y^n$ and substitute) gives:

$$\frac{dv}{dx} + (1-n)P(x)\,v = (1-n)Q(x)$$

which is a standard **linear** ODE.

### Example 3.1 — Logistic Growth

$$\frac{dy}{dt} = r\,y\left(1 - \frac{y}{K}\right)$$

This is Bernoulli with $n=2$.  
Analytical solution: $y(t) = \dfrac{K\,y_0}{y_0 + (K - y_0)\,e^{-rt}}$

In [ ]:
def logistic(t, y0, r, K):
    return K * y0 / (y0 + (K - y0) * np.exp(-r * t))

t = np.linspace(0, 12, 400)
K, r = 100.0, 0.8
y0_values = [5, 15, 40, 80, 120, 160]

fig, ax = plt.subplots(figsize=(9, 5))
colors = cm.viridis(np.linspace(0.1, 0.9, len(y0_values)))

for y0, c in zip(y0_values, colors):
    ax.plot(t, logistic(t, y0, r, K), color=c, lw=2, label=f'$y_0={y0}$')

ax.axhline(K, color='crimson', ls='--', lw=1.5, label=f'Carrying capacity K={K}')
ax.set_xlabel('Time t'); ax.set_ylabel('Population y(t)')
ax.set_title('Logistic Growth — Bernoulli Equation (n=2)')
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); plt.show()

---

## 4. Exact Equations

The ODE $M(x,y)\,dx + N(x,y)\,dy = 0$ is **exact** if and only if:

$$\frac{\partial M}{\partial y} = \frac{\partial N}{\partial x}$$

If exact, there exists $F(x,y)$ with $\nabla F = (M, N)$. The general solution is $F(x,y) = C$.

**Finding F:**
1. Integrate $M$ with respect to $x$: $F = \int M\,dx + g(y)$
2. Differentiate $F$ with respect to $y$, set equal to $N$, solve for $g'(y)$
3. Integrate $g'(y)$ to get $g(y)$

In [ ]:
def check_exactness(M_func, N_func, x0=1.0, y0=1.0, h=1e-5):
    """Numerically verify exactness via finite-difference partial derivatives."""
    dM_dy = (M_func(x0, y0+h) - M_func(x0, y0-h)) / (2*h)
    dN_dx = (N_func(x0+h, y0) - N_func(x0-h, y0)) / (2*h)
    is_exact = np.isclose(dM_dy, dN_dx, atol=1e-6)
    print(f"  ∂M/∂y = {dM_dy:.8f}")
    print(f"  ∂N/∂x = {dN_dx:.8f}")
    print(f"  Exact: {is_exact}")
    return is_exact

# --- Case 1: EXACT ---
# (2xy + y^2)dx + (x^2 + 2xy)dy = 0
# F(x,y) = x^2*y + xy^2  =>  solution: x^2*y + xy^2 = C
print("Case 1: (2xy + y²)dx + (x² + 2xy)dy = 0")
M1 = lambda x, y: 2*x*y + y**2
N1 = lambda x, y: x**2 + 2*x*y
check_exactness(M1, N1)

print()

# --- Case 2: NOT EXACT ---
print("Case 2: x*dx + x*dy = 0  (not exact)")
M2 = lambda x, y: x
N2 = lambda x, y: x
check_exactness(M2, N2)

---

## 5. Direction Fields — From Scratch

A **direction field** (slope field) draws a unit tangent vector at each grid point $(x_i, y_j)$ with slope $f(x_i, y_j)$, giving a geometric portrait of the entire solution family.

In [ ]:
def direction_field(f, x_range, y_range, nx=24, ny=24, ax=None, color='#4a90d9', alpha=0.65):
    """Draw direction field for dy/dx = f(x, y) using normalised quiver arrows."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    x_vals = np.linspace(*x_range, nx)
    y_vals = np.linspace(*y_range, ny)
    X, Y = np.meshgrid(x_vals, y_vals)
    DX = np.ones_like(X)
    DY = f(X, Y)
    norm = np.sqrt(DX**2 + DY**2) + 1e-12
    DX /= norm; DY /= norm
    scale = (x_range[1] - x_range[0]) / nx * 0.75
    ax.quiver(X, Y, DX * scale, DY * scale,
              angles='xy', scale_units='xy', scale=1,
              color=color, alpha=alpha, width=0.003, headwidth=3)
    return ax

def euler_trace(f, x0, y0, x_end, h=0.05):
    """Forward Euler integration to overlay a specific solution curve."""
    xs, ys = [x0], [y0]
    x, y = x0, y0
    while x < x_end - h/2:
        y += h * f(x, y)
        x += h
        xs.append(x); ys.append(y)
    return np.array(xs), np.array(ys)


# --- Logistic ODE phase portrait ---
f_logistic = lambda x, y: y * (1 - y / 5.0)

fig, ax = plt.subplots(figsize=(9, 6))
direction_field(f_logistic, x_range=(0, 8), y_range=(-1, 8), ax=ax)

ic_values = [-0.5, 0.5, 1.5, 3.0, 5.0, 6.5, 7.5, 8.0]
colors = cm.plasma(np.linspace(0.1, 0.85, len(ic_values)))
for y0_ic, c in zip(ic_values, colors):
    xs, ys = euler_trace(f_logistic, x0=0, y0=y0_ic, x_end=8, h=0.02)
    ax.plot(xs, ys, color=c, lw=2)

ax.axhline(5, color='crimson', ls='--', lw=1.5, label='Stable eq. y=5')
ax.axhline(0, color='gray', ls='--', lw=1.0, label='Unstable eq. y=0')
ax.set_xlim(0, 8); ax.set_ylim(-1, 8.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title("Direction Field: $y' = y(1 - y/5)$")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 6. Homogeneous First-Order Equations

A first-order ODE is **homogeneous** (in the classical sense) if it can be written as:

$$\frac{dy}{dx} = F\!\left(\frac{y}{x}\right)$$

The key observation is that $F$ depends only on the **ratio** $v = y/x$. The substitution

$$v = \frac{y}{x} \implies y = vx \implies y' = v + x\,v'$$

converts the equation into a **separable** ODE in $v(x)$:

$$v + x\,v' = F(v) \implies \frac{dv}{F(v)-v} = \frac{dx}{x}$$

**Example:** $y' = \dfrac{x^2 + y^2}{xy}$

Divide numerator and denominator by $x^2$:
$$y' = \frac{1 + (y/x)^2}{y/x} = F(v),\quad v = y/x$$
After substitution and integration: $y^2 = x^2(2\ln|x| + C)$.

In [ ]:
def solve_homogeneous_ode(F, x0, y0, x_end, n=1000):
    """
    Solve a homogeneous first-order ODE  dy/dx = F(y/x)
    via the substitution v = y/x, reducing to separable form.
    Uses RK4 on the transformed equation:
        dv/dx = (F(v) - v) / x
    Parameters
    ----------
    F     : callable  F(v) where v = y/x
    x0,y0 : initial condition (x0 != 0)
    x_end : end of integration interval
    n     : number of steps
    """
    x_arr = np.linspace(x0, x_end, n)
    h = x_arr[1] - x_arr[0]
    v = y0 / x0          # initial v
    v_arr = [v]

    def dvdx(x, v):
        if abs(x) < 1e-14:
            return 0.0
        return (F(v) - v) / x

    for x in x_arr[:-1]:
        k1 = dvdx(x,         v)
        k2 = dvdx(x + h/2,   v + h*k1/2)
        k3 = dvdx(x + h/2,   v + h*k2/2)
        k4 = dvdx(x + h,     v + h*k3)
        v  = v + h*(k1 + 2*k2 + 2*k3 + k4)/6
        v_arr.append(v)

    v_arr = np.array(v_arr)
    y_arr = v_arr * x_arr        # recover y = v*x
    return x_arr, y_arr


# ── Example 1: y' = (x² + y²)/(xy)  →  F(v) = (1 + v²)/v ────────────────
def F_ex1(v):
    return (1 + v**2) / v

# Analytical solution: y² = x²(2*ln|x| + C),  C chosen by (x0,y0)
x0, y0 = 1.0, 1.0
C_ex1 = y0**2 / x0**2 - 2*np.log(abs(x0))   # = 1

x_num, y_num = solve_homogeneous_ode(F_ex1, x0, y0, x_end=3.0)
x_ana = np.linspace(x0, 3.0, 500)
y_ana = x_ana * np.sqrt(2*np.log(x_ana) + C_ex1)

# ── Example 2: y' = (y - x)/(y + x)  →  F(v) = (v-1)/(v+1) ─────────────
def F_ex2(v):
    return (v - 1) / (v + 1)

x0b, y0b = 1.0, 2.0
x_num2, y_num2 = solve_homogeneous_ode(F_ex2, x0b, y0b, x_end=4.0)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(x_num, y_num, lw=2.5, color='#e74c3c', label='RK4 (numerical)')
ax.plot(x_ana, y_ana, lw=1.5, ls='--', color='#2c3e50', label='Analytical')
ax.set_title(r"$y' = (x^2+y^2)/(xy)$", fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(x_num2, y_num2, lw=2.5, color='#3498db', label='RK4 (numerical)')
ax.set_title(r"$y' = (y-x)/(y+x)$", fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('Homogeneous First-Order ODEs — substitution $v = y/x$', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

max_err = np.max(np.abs(y_num - np.interp(x_num, x_ana, y_ana)))
print(f'Max absolute error vs analytical (Example 1): {max_err:.2e}')

---

## 7. Summary

| ODE Type | Identifying Feature | Method |
|----------|--------------------|---------|
| **Separable** | $y' = g(x)h(y)$ | Separate and integrate |
| **Linear** | $y' + P(x)y = Q(x)$ | Multiply by $\mu = e^{\int P\,dx}$ |
| **Bernoulli** | $y' + P(x)y = Q(x)y^n$ | Substitute $v = y^{1-n}$ |
| **Exact** | $\partial_y M = \partial_x N$ | Find potential $F(x,y)$ |
| **Homogeneous** | $y' = F(y/x)$ | Substitute $v = y/x$ → separable |

---

## Exercises

1. **(Separable)** Solve $y' = \dfrac{x^2}{1-y^2}$ with $y(0)=0$. Plot the solution and identify its domain of validity.

2. **(Linear)** Solve $y' - 2xy = e^{x^2}$ using the integrating factor. Verify your answer by substituting back.

3. **(Bernoulli)** Solve $y' = ay - by^2$ for general $a, b > 0$. How does the long-time behaviour depend on $a/b$?

4. **(Exact)** Determine if $(3x^2y + y^3)dx + (x^3 + 3xy^2)dy = 0$ is exact, and if so find the potential function $F(x,y)$.

5. **(Direction Field)** Write a direction field renderer that colours each arrow by the magnitude of the slope. Apply it to $y' = \sin(xy)$.